# MeshAPI Gateway Basics (bare minimum)

The absolute minimum needed to start talking to **MeshAPI** (`https://api.meshapi.ai`) -- an AI
model gateway: one API key, one OpenAI-shaped API, many providers behind it.

This notebook only covers four things:
1. Opening a client
2. One basic chat completion
3. Checking what that call cost (tokens + real $)
4. Closing the client

That's it. Streaming, `compare`, model discovery, tool calling, structured outputs, error handling,
and every other feature are covered properly in `features_lazy_imports.ipynb` -- no need to repeat
them here. Next up: `02_rag_multiagent_lazy_imports.ipynb` builds a real RAG + multi-agent app on top
of exactly what this notebook teaches, then `features_lazy_imports.ipynb` tours everything else.

## 1. Install

In [1]:
%pip install -q meshapi python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2
[notice] To update, run: C:\Users\djadh\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


## 2. Open a client

The SDK does **not** auto-read env vars -- you pass `base_url` and `token` explicitly. Get your
`rsk_...` key from the MeshAPI dashboard.

In [2]:
import os
from getpass import getpass

from dotenv import load_dotenv
load_dotenv()

from meshapi import MeshAPI

MESHAPI_TOKEN = os.getenv("MESH_API_KEY") or getpass("MeshAPI token (rsk_...): ")
client = MeshAPI(base_url=os.getenv("MESHAPI_BASE_URL", "https://api.meshapi.ai"), token=MESHAPI_TOKEN)
print("Client ready.")

Client ready.


## 3. A basic chat completion

One call: a `model` string, a list of messages, get a reply back. `model` is a `"provider/model"`
string -- swap it for any other provider MeshAPI supports and this exact code still works. That's
the entire pitch of a gateway.

(Handling a model that might not exist, or picking one live from the catalog, is covered later in
`02_rag_multiagent_lazy_imports.ipynb` and `features_lazy_imports.ipynb` -- kept out of this
notebook on purpose.)

In [3]:
from meshapi import ChatCompletionParams, ChatMessage

resp = client.chat.completions.create(
    ChatCompletionParams(
        model="openai/gpt-4o-mini",
        messages=[ChatMessage(role="user", content="In one sentence, what is an AI gateway?")],
        max_tokens=60,
    )
)
print(resp.choices[0].message.content)

An AI gateway is a platform or interface that facilitates the integration, management, and deployment of artificial intelligence services and applications across different systems and environments.


## 4. Check what that call actually cost

Every response includes `usage` -- exactly how many tokens the call spent. Combine that with the
model's published pricing (one lookup, `client.models.get(...)`, no need to fetch the whole catalog
for this) and you can print the real cost of any call, any time after it finishes.

In [4]:
usage = resp.usage
print(f"tokens used -- prompt: {usage.prompt_tokens}, completion: {usage.completion_tokens}, total: {usage.total_tokens}")

pricing = client.models.get("openai/gpt-4o-mini").pricing
cost_usd = (usage.prompt_tokens * float(pricing.prompt_usd_per_1m) + usage.completion_tokens * float(pricing.completion_usd_per_1m)) / 1_000_000
print(f"estimated cost of this call: ${cost_usd:.6f}")

tokens used -- prompt: 17, completion: 29, total: 46


estimated cost of this call: $0.000020


## 5. Close the client

In [5]:
client.close()
print("Done. Next: open 02_rag_multiagent_lazy_imports.ipynb")

Done. Next: open 02_rag_multiagent_lazy_imports.ipynb
